<a href="https://colab.research.google.com/github/Harshitaa0523/GenAI-for-Business-Fine-tuning-LLM/blob/main/The_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GenAI for Business Analysis - Fine-Tuning LLM


### Task 1 - Set up the project environment

Install OpenAI and python-dotenv Python Module

In [ ]:
!pip install openai python-dotenv --upgrade

Importing modules

In [ ]:
import pandas as pd
import os, time
from openai import OpenAI
from dotenv import load_dotenv
import json
import matplotlib.pyplot as plt


print("Modules are imported.")

Setting up the OpenAI API:

* Prepare a .env file to store the OpenAI API key.
* Uploading the .env file to our colab environment
* Load the API key and setup the API

Creating OpenAI Client

In [ ]:
from openai import OpenAI
from google.colab import userdata

# Ensure APIKEY_SECURE is loaded from google.colab.userdata
# If it's not yet loaded, you may need to run cell 6e4e104e first.
# Using APIKEY_SECURE for robust and secure API key management.
client = OpenAI(
    api_key=userdata.get('APIKEY')
)
client

### Task 2 - Prepare the training data

Loading the provided `Customer Complaints.csv`



In [ ]:
import pandas as pd
import os

file_name = "Customer Complaints.csv"

if not os.path.exists(file_name):
    print(f"Error: The file '{file_name}' was not found.")
    print("Please upload the file to your Colab environment (e.g., using the files icon on the left sidebar, or `from google.colab import files; files.upload()`) or ensure the correct path is provided if it's in Google Drive.")
    print("Once the file is uploaded, re-run this cell.")
else:
    training_data = pd.read_csv(file_name)
    print("File loaded successfully:")
    training_data

**Converting the Complaints records to json**

To be able to use the data for the fine-tuning purpose, we first need to convert each row of the dataframe into the following format:

<pre>
<code>
{
  <span style="color: blue;">"messages"</span>: [
    {
      <span style="color: blue;">"role"</span>: <span style="color: red;">"system"</span>,
      <span style="color: blue;">"content"</span>: "<span style="color: green;">Providing context about the user's prompt.
                  It may include information about the task,
                  instructions, or background details relevant
                  to the conversation.</span>"
    },
    {
      <span style="color: blue;">"role"</span>: <span style="color: red;">"user"</span>,
      <span style="color: blue;">"content"</span>: "<span style="color: green;">the prompt or input provided by the user,
                  which typically initiates the conversation with the assistant.</span>"
    },
    {
      <span style="color: blue;">"role"</span>: <span style="color: red;">"assistant"</span>,
      <span style="color: blue;">"content"</span>: "<span style="color: green;">The desired response or output generated by
                  the assistant in response to the user's prompt.</span>"
    }
  ]
}
</code>
</pre>


Define a method that get's a row of the dataframe and convert it into the json format

In [7]:
def save_as_json(row):

  system_content = """
      Given a customer complaint text, extract and return the following information in json (dict) format:
      - Topic: The product/department that the customer has a complaint about.
      - Problem: A two or three-word description of what exactly the problem is.
      - Customer_Dissatisfaction_Index: is a number between 0 and 100 showing
             how angry the customer is about the problem.
  """

  formatted_data = {
        "messages": [
            {"role": "system", "content": system_content},
            {"role": "user", "content": row.Complaints},
            {"role": "assistant", "content": row.Details}
        ]
      }

  with open("training_data.json", "a") as json_file:
        json.dump(formatted_data, json_file)
        json_file.write("\n")

Use of this method to generate the `training_data.json`

In [8]:
for index, row in training_data.iterrows():
  save_as_json(row)

### Task 3 - Fine-tune GPT 3.5 based on our training data

Import the json file prepared as the training data

In [9]:
data_file = client.files.create(
    file=open("training_data.json", "rb"),
  purpose='fine-tune'
)
data_file

FileObject(id='file-Ln9iTTL5rWVtkcD2Vj3z4M', bytes=46722, created_at=1779812542, filename='training_data.json', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

### Loading API Key Securely with Colab Secrets

For better security in Colab, it's recommended to use the built-in Secrets manager to store your API key. Once stored, you can load it as shown below:

Create the Fine Tuning Job

In [ ]:
fine_tuning_job = client.fine_tuning.jobs.create(
    training_file=data_file.id,
    model="gpt-3.5-turbo",
    hyperparameters={
        "n_epochs": 'auto'
    }
)
fine_tuning_job

Retrieve the state of the fine-tune

In [ ]:
while True:
  time.sleep(2)
  retrieved_job = client.fine_tuning.jobs.retrieve(fine_tuning_job.id)
  status = retrieved_job.status
  print(status)

  if(status == "completed"):
    print("The job is done!")
    break

### Task 4 - Evaluate model

Retrieve the event messages to check out the learning process of our fine-tuning job.

In [ ]:
events = list(client.fine_tuning.jobs.list_events(fine_tuning_job_id = retrieved_job.id, limit = 100 ))

for e in events:
  print(e.message)

Extract the training loss in each learning step

In [ ]:
steps = []
train_loss = []

for e in events:
  if(e.data):
    steps.append(e.data['step'])
    train_loss.append(e.data['training_loss'])

print(steps)
print(train_loss)

Use a line chart to visualize the train_loss in each step

In [ ]:
from matplotlib.lines import lineStyles
plt.plot(steps, train_loss, marker='o',linestyles='')


### Task 5 - Deploy our model

Let's take a look at `retrieved_job` again

In [10]:
myLLM = retrieved_job.fine_tuned_model
print(myLLM)

Defining a method to extract information from a given user complaint using a specific LLM and return the results.

In [11]:
def extract_details(user_complaint, model_name):
    """
    This function extracts information from a given user complaint using a specific LLM (Large Language Model).

    Parameters:
    user_complaint (str): The text of the user's complaint.
    model_name (str): The name of the specific LLM model to use for extraction.
    """

    system_content = """
        Given a customer complaint text, extract and return the following information in JSON (dict) format:
        - Topic
        - Problem
        - Customer_Dissatisfaction_Index
    """

    # Generate a response using the specified model and the user's complaint
    response = client.chat.completions.create(
        model = model_name,
        messages=[
            {"role": "system", "content": system_content},  # System content explaining the expected output
            {"role": "user", "content": user_complaint}  # User's complaint passed as content
        ]
    )

    # Return the content of the generated response
    return response.choices[0].message.content


Let's use our fine-tuned model to extract the details for the following user complaint:

*TV channels keep disappearing from my subscription! What's going on? Extremely annoyed with this service!*

In [11]:
complaint = "TV channels keep disappearing from my subscription! What's going on? Extremely annoyed with this service!"
print(extract_details(complaint, myLLM))

Let's test our `GPT-4` model with the same user complaint

In [11]:
extract_details(complaint, 'gpt-4')
print(extract_details(complaint, 'gpt-4'))

Let's try for the following complaint:

*Line is down! It is really annoying!*

In [11]:
complaint = "Line is down! It is really annoying!"
print(extract_details(complaint, myLLM))

Now let's compare the results from GPT-4

In [11]:
extract_details(complaint, 'gpt-4')
print(extract_details(complaint, 'gpt-4'))

We can see that our model, which is trained on our dataset, provides better answers compared to GPT-4. Our model is fine-tuned based on our data and is familiar with the different edge cases and the context of our dataset.

In [12]:
customer_complaint = "I am very Angry! I want my money back!"